# `ssleeg` — Quickstart Notebook

Interactive walkthrough of the **semi-supervised learning benchmark for EEG emotion recognition**.

Everything here runs on **synthetic data** (no downloads) and on **CPU**, so you can execute the whole notebook top-to-bottom in a couple of minutes. It covers:

1. Setup & listing available components
2. Peeking at the (synthetic) EEG data
3. Training one SSL method
4. Evaluation & metrics
5. Figures (confusion matrix, ROC, t-SNE, learning curves)
6. A small benchmark grid → tables
7. Label-efficiency plot
8. Statistical significance tests
9. Plugging in **your own method**
10. Switching to real datasets (DEAP/SEED)

> Run this notebook from the **repository root** (the folder containing `src/` and `configs/`).

## 1. Setup
Make the `src/` package importable and pull in what we need. Importing `ssleeg` populates the dataset/model/method registries.

In [ ]:
import os, sys

# Ensure we are at the repo root and that src/ is importable.
if os.path.isdir('../src') and not os.path.isdir('src'):
    os.chdir('..')              # in case the notebook is opened from notebooks/
sys.path.insert(0, os.path.abspath('src'))

%matplotlib inline
import torch
import numpy as np

import ssleeg  # registers all datasets / models / methods
from ssleeg.utils.registry import DATASETS, MODELS, METHODS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)

## 2. What's available?
Every component is selectable from config by name. Here's the catalogue.

In [ ]:
print('DATASETS:', DATASETS.keys())
print()
print('MODELS:  ', MODELS.keys())
print()
print('METHODS: ', METHODS.keys())

## 3. Peek at the synthetic EEG data
The built-in `synthetic` dataset fabricates class-conditional oscillatory signals (per-emotion frequency bands + spatial topographies, with subject-specific shifts), so the full pipeline is testable without real data.

In [ ]:
import matplotlib.pyplot as plt

ds = DATASETS.build('synthetic', num_classes=3, num_subjects=4, num_sessions=1,
                    trials_per_class_per_session=30, num_channels=16, num_timepoints=128, seed=0)
print(f'X shape: {ds.X.shape}  (trials, channels, time)')
print(f'classes: {ds.num_classes} {ds.class_names} | subjects: {len(np.unique(ds.subjects))}')

fig, axes = plt.subplots(1, ds.num_classes, figsize=(12, 2.6), sharey=True)
for c, ax in enumerate(axes):
    trial = ds.X[np.where(ds.y == c)[0][0]]
    ax.plot(trial[:4].T, lw=0.8)         # first 4 channels of one trial
    ax.set_title(f'class {c} ({ds.class_names[c]})')
    ax.set_xlabel('time')
axes[0].set_ylabel('amplitude')
fig.tight_layout(); fig.show()

## 4. Train one SSL method
We load the `smoke` experiment config (tiny synthetic setup), optionally override fields, and run the `Trainer`. A helper keeps things tidy for reuse below.

The labeled ratio is the fraction of the **training pool** that stays labeled; the rest becomes the unlabeled pool (no leakage into val/test).

In [ ]:
import logging
from ssleeg.utils.config import load_config, merge_overrides
from ssleeg.utils.seed import seed_everything
from ssleeg.utils.logging import setup_logger, get_logger
from ssleeg.engine.trainer import Trainer

BASE_CFG = 'configs/experiment/smoke.yaml'
OUT_ROOT = 'nb_outputs'

def train_run(overrides=None, seed=0, out_root=OUT_ROOT, device=DEVICE, quiet=False):
    """Train one configuration and return (trainer, output_dir, test_metrics)."""
    cfg = load_config(BASE_CFG)
    cfg = merge_overrides(cfg, (overrides or []) + [f'seed={seed}'])
    seed_everything(seed, deterministic=cfg.get('deterministic', True))
    out = os.path.join(out_root, cfg.data.name, cfg.method.name,
                       f'{cfg.model.name}_r{cfg.data.label_ratio}_s{seed}')
    setup_logger(out)
    if quiet:
        get_logger().setLevel(logging.WARNING)
    trainer = Trainer(cfg, output_dir=out, seed=seed, device=device)
    metrics = trainer.train()
    return trainer, out, metrics

trainer, run_dir, metrics = train_run(['method.name=fixmatch', 'data.label_ratio=0.1', 'train.epochs=5'])
print('\nTest metrics:', {k: round(v, 4) for k, v in metrics.items()})

## 5. Evaluate & inspect metrics
The `Trainer` already evaluated the best checkpoint on the held-out test set. We can also re-run evaluation explicitly to get raw predictions for figures.

In [ ]:
from ssleeg.engine.evaluator import evaluate_model
import pandas as pd

res = evaluate_model(trainer.method.eval_module(), trainer.test_loader, trainer.device, trainer.dm.num_classes)
pd.Series(res.metrics).round(4).to_frame('test')

## 6. Figures (inline)
Confusion matrix, ROC curves, learning curves, and a t-SNE of the learned features.

In [ ]:
from ssleeg.viz.plots import plot_confusion_matrix, plot_roc_curves, plot_learning_curves

names = trainer.dm.dataset.class_names
plot_confusion_matrix(res.labels, res.preds, names, title='FixMatch — Confusion Matrix');
plot_roc_curves(res.labels, res.probs, names, title='FixMatch — ROC');
plot_learning_curves(trainer.history, metrics=['val_accuracy', 'val_f1', 'val_balanced_accuracy']);

In [ ]:
from ssleeg.viz.embeddings import extract_features, plot_embeddings

feats, labels = extract_features(trainer.method.eval_module(), trainer.test_loader, trainer.device)
plot_embeddings(feats, labels, method='tsne', class_names=names, title='t-SNE of learned features');
# plot_embeddings(feats, labels, method='umap', ...)   # needs `pip install umap-learn`

## 7. A small benchmark grid → tables
Run a few methods × labeled ratios × seeds, then aggregate into a `mean ± std` table. Keep it tiny here; scale up for real results. (We silence per-epoch logs with `quiet=True`.)

In [ ]:
BENCH_ROOT = 'nb_outputs/bench'
methods = ['supervised', 'pseudo_label', 'mean_teacher', 'fixmatch', 'your_method']
ratios  = [0.05, 0.1, 0.2]
seeds   = [0, 1]

total = len(methods) * len(ratios) * len(seeds)
i = 0
for m in methods:
    for r in ratios:
        for s in seeds:
            i += 1
            print(f'[{i}/{total}] {m} | ratio={r} | seed={s}', end='  ->  ')
            _, _, mt = train_run([f'method.name={m}', f'data.label_ratio={r}', 'train.epochs=5'],
                                 seed=s, out_root=BENCH_ROOT, quiet=True)
            print(f"acc={mt['accuracy']:.3f}")
print('done.')

In [ ]:
from ssleeg.reporting.tables import collect_results, build_benchmark_table

df = collect_results(BENCH_ROOT)
table = build_benchmark_table(df, metric='accuracy', dataset='synthetic')
print('Accuracy (mean ± std %)')
table

In [ ]:
# Export the table in every publication format.
from ssleeg.reporting.tables import to_csv, to_latex, to_markdown
os.makedirs('nb_outputs/tables', exist_ok=True)
to_csv(table, 'nb_outputs/tables/synthetic_accuracy.csv')
print(to_markdown(table, title='synthetic — accuracy (%)'))
print('\nLaTeX:\n')
print(to_latex(table, caption='Accuracy on synthetic (mean ± std %).', label='tab:acc'))

## 8. Label-efficiency curve
Accuracy vs. labeled fraction for each method (mean ± std band).

In [ ]:
from collections import defaultdict
from ssleeg.viz.plots import plot_label_efficiency

by_method = defaultdict(lambda: defaultdict(list))
for _, row in df.iterrows():
    by_method[row['method']][row['label_ratio']].append(row['accuracy'])
plot_label_efficiency({m: dict(v) for m, v in by_method.items()}, metric_name='accuracy',
                      title='Label efficiency (synthetic)');

## 9. Statistical significance
Descriptive stats, paired tests vs a reference method, and a Friedman omnibus + average ranks across all conditions.

In [ ]:
from ssleeg.metrics.statistics import mean_std, confidence_interval, paired_ttest, wilcoxon_test, friedman_test, average_ranks

REF = 'your_method'
metric = 'accuracy'
print(f'=== {metric}: mean ± std (95% CI), and paired tests vs "{REF}" ===\n')
for (d, r), g in df.groupby(['dataset', 'label_ratio']):
    print(f'[{d} @ {int(r*100)}%]')
    per = {m: mg[metric].values for m, mg in g.groupby('method')}
    for m, vals in per.items():
        mu, sd = mean_std(vals); lo, hi = confidence_interval(vals)
        print(f'  {m:14s} {mu*100:5.2f} ± {sd*100:4.2f}  CI[{lo*100:.1f},{hi*100:.1f}]')
    if REF in per:
        for m, vals in per.items():
            if m == REF or len(vals) != len(per[REF]):
                continue
            tt = paired_ttest(per[REF], vals); wx = wilcoxon_test(per[REF], vals)
            print(f'    {REF} vs {m:14s} t-test p={tt["p_value"]:.3f}  wilcoxon p={wx["p_value"]:.3f}')
    print()

In [ ]:
# Friedman + average ranks across (dataset, ratio) conditions.
pivot = df.groupby(['method', 'dataset', 'label_ratio'])[metric].mean().reset_index()
conds = sorted(set(map(tuple, pivot[['dataset', 'label_ratio']].values.tolist())))
scores = {m: [float(pivot[(pivot.method==m) & (pivot.dataset==d) & (pivot.label_ratio==r)][metric].mean())
              for (d, r) in conds] for m in sorted(pivot.method.unique())}
complete = {m: v for m, v in scores.items() if not any(np.isnan(v))}
if len(complete) >= 3:
    fr = friedman_test(complete); ranks = average_ranks(complete)
    print(f"Friedman chi2={fr['statistic']:.3f}, p={fr['p_value']:.4f}\n")
    print('Average ranks (lower = better):')
    for m in sorted(ranks, key=ranks.get):
        print(f'  {m:14s} {ranks[m]:.3f}')

## 10. Plug in **your own method**
Edit [`src/ssleeg/methods/your_method.py`](../src/ssleeg/methods/your_method.py) and implement `compute_loss(self, labeled, unlabeled, step) -> (loss, logs)`:

- `labeled`   = `{'x': (B,C,T), 'y': (B,)}`
- `unlabeled` = `{'weak': (B',C,T), 'strong': (B',C,T), 'index', 'y'}`  *(do not train on `unlabeled['y']`)*
- `self.model(x)` → logits;  `self.model(x, return_features=True)` → `(logits, feats)`;  `self.model.project(x)` → contrastive embedding
- helpers: `self._sup_loss(batch)`, `consistency_weight(step, w, rampup)`

Then just re-run with `method.name=your_method` — it's compared under identical splits/backbones/seeds. After editing the file, restart the kernel (or use the autoreload cell below) so the change is picked up.

In [ ]:
# Quick sanity check: instantiate your method and run one loss step.
from ssleeg.models.base import build_model
from ssleeg.utils.config import Config

x = torch.randn(8, 16, 128)
labeled   = {'x': x, 'y': torch.randint(0, 3, (8,))}
unlabeled = {'weak': x, 'strong': x, 'y': torch.randint(0, 3, (8,)), 'index': torch.arange(8)}
model  = build_model(Config({'name': 'eegnet', 'args': {'kernel_length': 32}}), 16, 128, 3)
method = METHODS.build('your_method', model=model, cfg=Config({'name': 'your_method'}),
                       num_classes=3, device=torch.device('cpu'), total_steps=100)
loss, logs = method.compute_loss(labeled, unlabeled, step=1)
print('loss =', float(loss), '| logs =', {k: round(v, 4) for k, v in logs.items()})

## 11. Switching to real datasets (DEAP / SEED)
Download the dataset (license required — not bundled) and point the loader `root` at it. Everything else stays the same.

```python
cfg = load_config('configs/experiment/deap_fixmatch.yaml')
cfg = merge_overrides(cfg, ['data.loader.root=/path/to/DEAP/data_preprocessed_python',
                            'data.label_ratio=0.1', 'train.epochs=100'])
seed_everything(0)
trainer = Trainer(cfg, output_dir='outputs/deap_demo', seed=0, device=DEVICE)
trainer.train()
```

Use `data.protocol = subject` (leave-subjects-out) for cross-subject generalization, or `session` for cross-session. See `docs/DATASETS.md` for layouts and `docs/EXPERIMENTS.md` for a command for every workflow.

---
**Tip:** the same steps are available as CLI tools — `ssleeg-train`, `ssleeg-benchmark`, `ssleeg-visualize`, `ssleeg-stats` — for batch/headless runs.